# 01 — Prepare data (CPU notebook)

**Settings (right sidebar):** Accelerator = **None**, Internet = **On**, Persistence = Files only.

Streams ~800M tokens of Python from `codeparrot/codeparrot-clean-train`, tokenizes with the
codeparrot 32k BPE tokenizer and writes `train.bin` / `val.bin` (uint16) to `/kaggle/working`.
Takes roughly 30–60 min. **Uses 0 GPU hours.**

When it finishes: **Save & Run All**, then on the finished version open the *Output* tab →
**New Dataset** → name it `pygpt-python-tokens`. Notebook 02 reads from that dataset.

In [ ]:
# --- Get the code into /kaggle/working/pygpt ---------------------------------
# Primary: clone the public repo (always current). Fallback: the "pygpt-code"
# Kaggle Dataset (works with Internet off / if GitHub is unreachable).
REPO_URL = "https://github.com/Rehan123-bash/pygpt.git"
import os, shutil, subprocess
os.chdir("/kaggle/working")
if os.path.exists("pygpt"):
    shutil.rmtree("pygpt")
if REPO_URL:
    try:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "pygpt"], check=True)
    except Exception as e:
        print("clone failed, falling back to the pygpt-code dataset:", e)
if not os.path.exists("pygpt/model.py"):
    src = "/kaggle/input/pygpt-code"
    for c in (os.path.join(src, "pygpt"), src):
        if os.path.exists(os.path.join(c, "model.py")):
            shutil.copytree(c, "pygpt"); break
    else:
        raise SystemExit("no code: clone failed and no 'pygpt-code' dataset input")
print(sorted(os.listdir("pygpt")))

In [ ]:
%pip install -q datasets
import datasets, transformers; print("datasets", datasets.__version__, "transformers", transformers.__version__)

In [ ]:
# Run 2 (Chinchilla-ish): 2.2B fresh tokens. First run used 800_000_000.
TRAIN_TOKENS = 2_200_000_000
VAL_TOKENS   = 5_000_000
!cd /kaggle/working/pygpt && python data/prepare_data.py --out_dir /kaggle/working \
    --train_tokens {TRAIN_TOKENS} --val_tokens {VAL_TOKENS}

In [ ]:
# Blend the DSA curriculum at ~7% by appending the small corpus repeatedly.
# Random 512-token windows start uniformly over the file, so a contiguous block
# of N tokens simply gets sampled N/total of the time - no interleaving needed.
DSA_DIR = "/kaggle/input/pygpt-algo-tokens"
DSA_REPEAT = 10                       # 15.6M x 10 = 156M of ~2.36B = ~6.6%
import os
if DSA_REPEAT and os.path.exists(DSA_DIR + "/train.bin"):
    import numpy as np
    dsa = np.fromfile(DSA_DIR + "/train.bin", dtype=np.uint16)
    with open("/kaggle/working/train.bin", "ab") as f:
        for _ in range(DSA_REPEAT):
            dsa.tofile(f)
    sz = os.path.getsize("/kaggle/working/train.bin")
    print(f"appended DSA x{DSA_REPEAT} (+{len(dsa)*DSA_REPEAT/1e6:.0f}M tokens); "
          f"train.bin now {sz/2**30:.2f} GB = {sz//2:,} tokens")
else:
    print("pygpt-algo-tokens not mounted - blend skipped")

In [ ]:
import json, os
print(json.dumps(json.load(open("/kaggle/working/meta.json")), indent=2))
for f in ("train.bin", "val.bin"):
    print(f, f"{os.path.getsize('/kaggle/working/'+f)/2**20:,.0f} MB")
print(open("/kaggle/working/sample.txt").read()[:1500])

### Next
1. Confirm the sample above looks like real Python.
2. Output tab → **New Dataset** → `pygpt-python-tokens` (contains `train.bin`, `val.bin`, `meta.json`, `tokenizer/`).
3. Open Notebook 02.